# CRUX Step 5: Architecture A, Architecture B (Rule Compiler), the B-ablation, and the A-param control

This notebook runs the full plan on a Colab T4: 4 architectures x 6 trials x 3 seeds = 72 runs, with `budget.training_steps` steps each (`configs/protocol.yaml`). Every run saves its files to Google Drive. After a disconnect, run the cells again: finished runs are skipped, and an unfinished run resumes from its last checkpoint (saved every `eval_every` steps). The notebook reads the train, dev_id, and validation splits only. It never reads the test split.

In [ ]:
# Mount Drive, clone the repo, and install the requirements.
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/0xAnto/crux.git /content/crux
%cd /content/crux
!pip install -q -r requirements.txt

The splits are not in git. This cell makes them again from the generator and the benchmark converters (as in `notebooks/step4_splits.ipynb`). The build stops with an error if a file has another SHA-256 than the frozen manifest. Then `load_split` checks every split against `configs/splits_manifest.json` and `configs/protocol.yaml` again.

In [ ]:
!python -m crux.data.boardgameqa
!python -m crux.data.dynabench
!python -m crux.data.supernat
!python -m crux.gen.generator
!python -m crux.splits build
!python -m crux.splits verify
from crux.splits import load_split
for name in ('train', 'dev_id', 'validation'):
    print(name, len(load_split(name)))  # raises ValueError if the hash differs

The timing run (do this before the sweep). It trains A and B-abl (the slowest architecture) for 200 steps on a small slice, and checks the fp16 path on CUDA. The rate uses the steps after step 50, so the CUDA warmup does not count. Set `budget.training_steps` in `configs/protocol.yaml` from this rate, commit it, and pull again. The sweep refuses to start while the value is `null`.

In [ ]:
import json, subprocess
SMOKE = '/content/smoke_step5'
for arch in ('A', 'B-abl'):
    subprocess.run(['python', '-m', 'crux.train', '--config', 'configs/step5.yaml', '--arch', arch, '--trial', '0',
                    '--out', f'{SMOKE}/{arch}', '--set', 'train.steps=200', 'train.log_every=50',
                    'data.max_train=8000', 'data.max_eval=500'], check=True)
    log = [json.loads(l) for l in open(f'{SMOKE}/{arch}/train.log') if '"seconds"' in l]
    a, b = log[0], log[-1]
    rate = (b['seconds'] - a['seconds']) / (b['step'] - a['step'])
    print(f'{arch}: {rate:.3f} s per step (lower is better); 72 runs x 1000 steps = {72 * 1000 * rate / 3600:.1f} h')


The sweep. `RUNS` is on Drive. A run is one (architecture, trial, seed index). The order is trial first, so that each finished trial has all 3 seeds. Do not select a trial before all its seeds are done: the aggregation does this.

In [ ]:
import subprocess, yaml
RUNS = '/content/drive/MyDrive/crux/runs/step5'
cfg = yaml.safe_load(open('configs/step5.yaml'))
protocol = yaml.safe_load(open('configs/protocol.yaml'))
print('steps per run:', protocol['budget']['training_steps'])
ARCHS = list(cfg['architectures'])           # A, B, B-abl, A-param
TRIALS = range(len(cfg['search']['trials']))  # 6 trials
SEEDS = range(len(protocol['seeds']))         # 3 seeds
for trial in TRIALS:
    for arch in ARCHS:
        for seed in SEEDS:
            subprocess.run(['python', '-m', 'crux.train', '--config', 'configs/step5.yaml', '--arch', arch,
                            '--trial', str(trial), '--seed-index', str(seed),
                            '--out', f'{RUNS}/{arch}/t{trial}/s{seed}'], check=True)

The aggregation. For each architecture it selects the trial with the best mean validation `unseen_rule_accuracy` over the 3 seeds. It prints the mean and the standard deviation over the seeds, the parameter counts, the latency, and the 95% paired bootstrap CI of the difference to A. The tables name the better direction in each column.

In [ ]:
!python -m crux.aggregate {RUNS}